# 1 - Imports

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [3]:
from src.utils import config, io

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


In [4]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score

# 2 - Preprocessing

In [5]:
X = io.load_csv(config.PROCESSED_DATA_DIR / 'X.csv', index_col=0)
y = io.load_csv(config.PROCESSED_DATA_DIR / 'y.csv', index_col=0)

In [34]:
split_cfg = io.load_json(config.PROCESSED_DATA_DIR / 'splits/temporal_v1.json')
split_cfg

{'description': 'Forecasting split for future country risk prediction',
 'train_years': [1999, 2015],
 'val_years': [2016, 2018],
 'test_years': [2019, 2024]}

In [7]:
def get_subset_data(data, bounds):
    return data[(data['YEAR'] >= bounds[0]) & (data['YEAR'] <= bounds[1])]

In [ ]:
X_train = get_subset_data(X, split_cfg['train_years'])
y_train = y.loc[X_train.index]
X_val = get_subset_data(X, split_cfg['val_years'])
y_val = y.loc[X_val.index]
X_test = get_subset_data(X, split_cfg['test_years'])
y_test = y.loc[X_test.index]

# 3 - Train Model

In [ ]:
from src.models import baseline_lr, xgboost_model, torch_model, evaluate
from src.preprocessing import preprocess_pipeline

In [31]:
preprocessor = preprocess_pipeline.build_preprocessor(X)

In [30]:
import mlflow

mlflow.set_tracking_uri(config.PROJECT_ROOT / 'models/mlruns')
mlflow.set_experiment('Country Risk Prediction')


<Experiment: artifact_location='file:///Users/hippolytegrandet/Desktop/Dev/country_risk_rating/models/mlruns/991689756472581023', creation_time=1770566631957, experiment_id='991689756472581023', last_update_time=1770566631957, lifecycle_stage='active', name='Country Risk Prediction', tags={}>

## 3.1 - Baseline, Logistic Regression Model

In [ ]:
model_name = 'logistic_regression'

model_params = {
    'C': 1.0,
    'max_iter': 1000,
    'class_weight': 'balanced'
}

model = baseline_lr.get_model(
    preprocessor,
    model_params
)

In [44]:
with mlflow.start_run(run_name='baseline_lr_v1'):

    # Log split metadata
    mlflow.log_params({
        'model': model_name,
        **model_params,
        'train_years': split_cfg['train_years'],
        'val_years': split_cfg['val_years'],
        'test_years': split_cfg['test_years']
    })

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    val_metrics = evaluate.evaluate_model(model, X_val, y_val, prefix='val_')
    test_metrics = evaluate.evaluate_model(model, X_test, y_test, prefix='test_')

    mlflow.log_metrics({**val_metrics, **test_metrics})

    # Log model
    mlflow.sklearn.log_model(
        model,
        artifact_path='model',
        registered_model_name=None,
        input_example=X_test.loc[[X_test.index[0]]]
    )

    run_id = mlflow.active_run().info.run_id

print('MLflow run_id:', run_id)

2026/02/08 17:19:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validat

Baseline Logistic Regression Results
val_accuracy: 0.6839
val_precision: 0.6272
val_recall: 0.6372
val_f1: 0.6255

Classification Report
              precision    recall  f1-score   support

           1       0.93      0.69      0.79       162
           2       0.67      0.58      0.62        24
           3       0.55      0.72      0.62        46
           4       0.46      0.42      0.44        31
           5       0.45      0.60      0.52        40
           6       0.56      0.65      0.60        94
           7       0.76      0.80      0.78       125

    accuracy                           0.68       522
   macro avg       0.63      0.64      0.63       522
weighted avg       0.71      0.68      0.69       522


Confusion Matrix
[[112   1   2   4   1  21  21]
 [  2  14   8   0   0   0   0]
 [  1   6  33   6   0   0   0]
 [  1   0  13  13   4   0   0]
 [  0   0   4   1  24  10   1]
 [  0   0   0   1  23  61   9]
 [  4   0   0   3   1  17 100]]


2026/02/08 17:19:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Baseline Logistic Regression Results
test_accuracy: 0.6090
test_precision: 0.5080
test_recall: 0.4956
test_f1: 0.4931

Classification Report
              precision    recall  f1-score   support

           1       0.78      0.70      0.74       270
           2       0.50      0.33      0.40        33
           3       0.53      0.64      0.58        73
           4       0.12      0.07      0.08        46
           5       0.44      0.36      0.39        84
           6       0.44      0.67      0.53       139
           7       0.74      0.70      0.72       217

    accuracy                           0.61       862
   macro avg       0.51      0.50      0.49       862
weighted avg       0.62      0.61      0.61       862


Confusion Matrix
[[189   1   2   8   0  33  37]
 [ 11  11  11   0   0   0   0]
 [ 13   6  47   2   2   3   0]
 [  6   1  15   3  15   6   0]
 [  4   2   7   8  30  32   1]
 [  6   1   3   3  17  93  16]
 [ 12   0   3   1   4  45 152]]


/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in

MLflow run_id: 19e7ce582e774588a933156a2aadfff2


### Save Model to Registry

In [ ]:
from src.models import registry

In [ ]:
registry.save_to_registry(
    model, 
    model_type='baseline_lr', 
    test_metrics=test_metrics, 
    run_id=run_id
)

## 3.2 XGBoost Classifier

In [13]:
import xgboost as xgb

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <FBD6AEF9-AFAB-39D7-B881-755157DA0497> /Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/usr/local/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/

In [ ]:
clf = xgb.XGBClassifier(
    reg_alpha= 0.01,
    colsample_bytree=0.60,
    eta=0.3,
    eval_metric=['mlogloss'],
    gamma=0.00001,
    reg_lambda=1.04,
    max_depth=6,
    min_child_weight=0.2,
    num_class=7,
    objective='multi:softprob',
    subsample=0.73
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(accuracy_score(y_test, y_pred))

In [ ]:
lr_model_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression())
])

In [ ]:
lr_model_pipeline.fit(X_train, y_train)

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   KNNImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['YEAR', 'GE.EST', 'NY.GDP.PCAP.CD', 'DT.DOD.DLXF.CD', 'DT.DOD.DIMF.CD',
       'SP.URB.TOTL.IN.ZS', 'SP.POP.TOTL', 'FS.AST.PRVT.GD.ZS',
       'DT.DOD.DECT.GN.ZS', 'FR.INR.LEND', 'EN.URB.LCTY.UR.ZS', 'SP.POP.DPND',
       'NY.GDP.MKT...
       'NV.SRV.TOTL.KD.ZG', 'NY.GNP.MKTP.KD.ZG', 'NY.GDP.MKTP.KD.ZG',
       'BX.KLT.DINV.WD.GD.ZS', 'NV.IND.TOTL.KD.ZG'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['ISO3_COUNTRY_CODE', 'GEO_REGION', 'ADMIN_REGION', 'LENDING_TYPE',
       'INCOME_GROUP'],
      dtype='object'))])),
                ('model', LogisticRegression())])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


In [ ]:
# Predictions
y_pred = lr_model_pipeline.predict(X_test)
y_proba = lr_model_pipeline.predict_proba(X_test)[:, 1]

# Evaluation metrics
results = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred, average='micro'),
    'recall': recall_score(y_test, y_pred, average='micro'),
    'f1': f1_score(y_test, y_pred, average='micro'),
    # 'roc_auc': roc_auc_score(y_test, y_proba, multi_class='ovr')
}

print('Baseline Logistic Regression Results')
for k, v in results.items():
    print(f'{k}: {v:.4f}')

print('\nClassification Report')
print(classification_report(y_test, y_pred))

print('\nConfusion Matrix')
print(confusion_matrix(y_test, y_pred))

Baseline Logistic Regression Results
accuracy: 0.7707
precision: 0.7707
recall: 0.7707
f1: 0.7707

Classification Report
              precision    recall  f1-score   support

           1       0.89      0.92      0.90       226
           2       0.81      0.82      0.81        77
           3       0.69      0.77      0.73        94
           4       0.59      0.55      0.57        66
           5       0.67      0.35      0.46       103
           6       0.62      0.64      0.63       166
           7       0.83      0.91      0.87       319

    accuracy                           0.77      1051
   macro avg       0.73      0.71      0.71      1051
weighted avg       0.76      0.77      0.76      1051


Confusion Matrix
[[208   8   0   1   0   2   7]
 [  6  63   8   0   0   0   0]
 [  7   7  72   5   1   0   2]
 [  3   0  16  36   6   4   1]
 [  4   0   6  11  36  35  11]
 [  3   0   2   8   9 106  38]
 [  4   0   0   0   2  24 289]]
